<a href="https://colab.research.google.com/github/groovesaido/Titanic-Survival-Prediction-ML-Project-classification-/blob/main/titanic_dataset_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install opendatasets

In [4]:
import opendatasets as od
import pandas as pd

In [6]:
#data loading
data= pd.read_csv('Titanic-Dataset.csv')

In [14]:
data['HasCabin']=data['Cabin'].notnull().astype(int)

In [17]:
data['Deck']=data['Cabin'].fillna("U").str[0]

In [ ]:
data['Deck'].unique()

In [32]:
# x(features) y(labels)
x=data.drop(['PassengerId','Survived','Ticket','Cabin'],axis=1)
y=data['Survived']

In [35]:
#data split train
from sklearn.model_selection import train_test_split
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.1,random_state=42)

In [42]:
#encoding from categorical to numerical
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output= False,
                        handle_unknown='ignore'
                        ).set_output(transform="pandas")
x_train_encoded=encoder.fit_transform(x_train[['Deck','Sex','Embarked']])
x_test_encoded=encoder.transform(x_test[['Deck','Sex','Embarked']])

In [46]:
#concatination
x_train=pd.concat([x_train,x_train_encoded],axis=1).drop(columns=['Deck','Sex','Embarked'])
x_test=pd.concat([x_test,x_test_encoded],axis=1).drop(columns=['Deck','Sex','Embarked'])

In [51]:
x_train['Agemissing']=x_train['Age'].isnull().astype(int)
x_test['Agemissing']=x_test['Age'].isnull().astype(int)

In [ ]:
x_train['Age']=x_train['Age'].fillna(x_train['Age'].median())
x_test['Age']=x_test['Age'].fillna(x_test['Age'].median())

In [55]:
x_train=x_train.drop(columns=['Embarked_nan'],axis=1)
x_test=x_test.drop(columns=['Embarked_nan'],axis=1)

In [58]:
#scaling
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.transform(x_test)


In [69]:
#model
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=500, random_state=42)
model.fit(x_train,y_train)

RandomForestClassifier(n_estimators=500, random_state=42)

In [97]:
#testing
print(f"Accuracy: {round(model.score(x_test,y_test)*100)}%")

Accuracy: 81%


In [99]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Ticket       891 non-null    object 
 8   Fare         891 non-null    float64
 9   Cabin        204 non-null    object 
 10  Embarked     889 non-null    object 
 11  HasCabin     891 non-null    int64  
 12  Deck         891 non-null    object 
dtypes: float64(2), int64(6), object(5)
memory usage: 90.6+ KB


In [ ]:
#feature importance
feature_importance = pd.DataFrame({
    "Feature": ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_U', 'Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Agemissing'],
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance

In [ ]:
#predicting a single instance
sample_pos=11
single_instance = x_test[sample_pos].reshape(1, -1)
model.predict(single_instance)
print(f"Predicted: {model.predict(single_instance).item()}, actual: {y_test.iloc[sample_pos]}")